# 03_RQ2_SHARED_CONTEXT_ROUTING_v2_1

**Trương Minh Hoàng · MSSV 25410008**

Notebook nghiên cứu KLTN v2.1. Đọc [hướng dẫn chạy](README.md) và chuẩn bị dữ liệu trước khi thực thi. Output Colab cũ được bỏ; kết quả đã khóa nằm ở [evidence/historical](../evidence/historical/).

Research notebook: dataset-backed execution is separate from the small public demonstrations. Scientific feature formulas, model candidates and routing checks are retained. Public packaging changes cover paths, optional installation and fresh-run defaults.

# 03 — RQ2: Shared Risk Context và định tuyến gọn (v2.1)

Notebook này **không xây joint Credit–Fraud model, không liên kết khách hàng giữa LendingClub và DGraphFin, không tạo joint score**.

Mục tiêu RQ2:

1. Chạy adapter trên toàn bộ source outputs đã khóa của Credit, DGraphFin node-only và DGraphFin node+graph.
2. Kiểm tra conformance theo semantic contract đã khai báo: domain, score, score semantics, entity scope, model/source version và provenance.
3. Định tuyến đơn giản theo miền khi context hợp lệ; lỗi đã xác định bị từ chối, nguồn tham chiếu tạm thời không sẵn sàng được `HELD → DATA_REVIEW`.
4. Kiểm tra correctness, determinism, traceability và cấm joint score.


In [ ]:
# Install the optional research requirements before running.
# %pip -q install pyarrow

In [ ]:
import os
from IPython.display import display
from pathlib import Path
from copy import deepcopy
import hashlib, json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

POINTER_PATH = Path(os.environ.get('KLTN_CONFIG_POINTER', '/content/drive/MyDrive/KLTN_REPRODUCTION_CONFIG_POINTER.json' if Path('/content/drive/MyDrive').is_dir() else str(Path.cwd() / 'KLTN_REPRODUCTION_CONFIG_POINTER.json'))).resolve()
if not POINTER_PATH.exists():
    raise FileNotFoundError('Chưa có config pointer. Hãy chạy notebook 00 trước.')
CONFIG_PATH = Path(json.loads(POINTER_PATH.read_text(encoding='utf-8'))['config_path'])
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

CREDIT_DIR = Path(CONFIG['credit_dir'])
RQ1_DIR = Path(CONFIG['rq1_dir'])
RQ2_DIR = Path(CONFIG['rq2_dir']); RQ2_DIR.mkdir(parents=True, exist_ok=True)

CREDIT_SOURCE = CREDIT_DIR / 'credit_predictions_for_context.parquet'
FRAUD_NODE_SOURCE = RQ1_DIR / 'dgraphfin_node_only_predictions_for_context.parquet'
FRAUD_GRAPH_SOURCE = RQ1_DIR / 'dgraphfin_node_plus_graph_predictions_for_context.parquet'
CREDIT_MANIFEST_PATH = CREDIT_DIR / 'credit_evidence_manifest.json'
RQ1_MANIFEST_PATH = RQ1_DIR / 'rq1_final_manifest.json'
for path in [CREDIT_SOURCE, FRAUD_NODE_SOURCE, FRAUD_GRAPH_SOURCE, CREDIT_MANIFEST_PATH, RQ1_MANIFEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Thiếu source/evidence artifact đã khóa: {path}')
print('RQ2 inputs + upstream manifests ready.')


In [ ]:
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_hash(obj) -> str:
    payload = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(',', ':'), default=str).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

credit_manifest = json.loads(CREDIT_MANIFEST_PATH.read_text(encoding='utf-8'))
rq1_manifest = json.loads(RQ1_MANIFEST_PATH.read_text(encoding='utf-8'))
if credit_manifest.get('status') != 'CREDIT_SOURCE_READY':
    raise RuntimeError('Credit source manifest status không hợp lệ.')
if rq1_manifest.get('status') != 'RQ1_FINAL_EVIDENCE_LOCKED':
    raise RuntimeError('RQ1 final manifest status không hợp lệ.')
for path in [CREDIT_SOURCE, FRAUD_NODE_SOURCE, FRAUD_GRAPH_SOURCE]:
    expected = None
    if path == CREDIT_SOURCE:
        expected = credit_manifest['files'].get(path.name)
    else:
        expected = rq1_manifest['files'].get(path.name)
    actual = sha256_file(path)
    if expected != actual:
        raise RuntimeError(f'Upstream artifact hash mismatch: {path.name}')

CONTEXT_VERSION = 'shared_risk_context_v2.1'
CREDIT_ADAPTER_VERSION = 'credit_adapter_v2.1'
FRAUD_ADAPTER_VERSION = 'fraud_graph_adapter_v2.1'
ROUTING_POLICY_VERSION = 'routing_policy_v2.1'

SOURCE_REGISTRY = {
    'credit': {
        'path': CREDIT_SOURCE,
        'artifact_sha256': sha256_file(CREDIT_SOURCE),
        'entity_scope': 'lendingclub_loan_record',
        'risk_domain': 'credit',
        'score_semantics': 'lightgbm_default_class_score',
        'source_dataset': 'lendingclub_granting_models_v0.1',
        'model_versions': {'credit_lightgbm_v2.1'},
        'assessment_time_status': 'source_observed_month',
        'adapter_version': CREDIT_ADAPTER_VERSION,
        'route': 'CREDIT_REVIEW',
    },
    'fraud_node_only': {
        'path': FRAUD_NODE_SOURCE,
        'artifact_sha256': sha256_file(FRAUD_NODE_SOURCE),
        'entity_scope': 'dgraphfin_user_node',
        'risk_domain': 'fraud_graph',
        'score_semantics': 'dgraphfin_anomaly_class_score',
        'source_dataset': 'dgraphfin_original',
        'model_versions': {'dgraphfin_lightgbm_node_only_v2.1'},
        'assessment_time_status': 'not_available_in_source',
        'adapter_version': FRAUD_ADAPTER_VERSION,
        'route': 'FRAUD_GRAPH_REVIEW',
    },
    'fraud_node_plus_graph': {
        'path': FRAUD_GRAPH_SOURCE,
        'artifact_sha256': sha256_file(FRAUD_GRAPH_SOURCE),
        'entity_scope': 'dgraphfin_user_node',
        'risk_domain': 'fraud_graph',
        'score_semantics': 'dgraphfin_anomaly_class_score',
        'source_dataset': 'dgraphfin_original',
        'model_versions': {'dgraphfin_lightgbm_node_plus_graph_v2.1'},
        'assessment_time_status': 'not_available_in_source',
        'adapter_version': FRAUD_ADAPTER_VERSION,
        'route': 'FRAUD_GRAPH_REVIEW',
    },
}

SOURCE_REQUIRED = {
    'source_record_reference', 'entity_scope', 'risk_domain', 'score_value',
    'score_semantics', 'model_version', 'source_dataset',
    'assessment_time', 'assessment_time_status'
}
CONTEXT_REQUIRED = SOURCE_REQUIRED | {
    'context_version', 'adapter_version', 'source_artifact_sha256',
    'trace_id', 'quality_flags'
}

CONTRACT = {
    'context_version': CONTEXT_VERSION,
    'score_range': [0.0, 1.0],
    'joint_score_allowed': False,
    'real_customer_linkage_claim_allowed': False,
    'routing_scope': 'offline_recommendation_only',
    'required_fields': sorted(CONTEXT_REQUIRED),
    'optional_fields': ['reason_signals', 'risk_band', 'risk_band_policy'],
}
CONTRACT_SHA256 = canonical_hash(CONTRACT)
ROUTING_POLICY = {
    'policy_version': ROUTING_POLICY_VERSION,
    'single_context_routes': {'credit': 'CREDIT_REVIEW', 'fraud_graph': 'FRAUD_GRAPH_REVIEW'},
    'held_route': 'DATA_REVIEW',
    'rejected_route': 'NONE',
    'controlled_multi_context_route': 'MULTI_CONTEXT_REVIEW',
    'decision_authority': 'NONE_RECOMMENDATION_ONLY',
}
ROUTING_POLICY_SHA256 = canonical_hash(ROUTING_POLICY)
(RQ2_DIR / 'shared_risk_context_contract_v2_1.json').write_text(json.dumps({**CONTRACT, 'contract_sha256': CONTRACT_SHA256}, ensure_ascii=False, indent=2), encoding='utf-8')
(RQ2_DIR / 'routing_policy_v2_1.json').write_text(json.dumps({**ROUTING_POLICY, 'policy_sha256': ROUTING_POLICY_SHA256}, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'contract_sha256': CONTRACT_SHA256, 'routing_policy_sha256': ROUTING_POLICY_SHA256}, ensure_ascii=False, indent=2))


In [ ]:
def adapt_record(record: dict, source_key: str) -> dict:
    spec = SOURCE_REGISTRY[source_key]
    # Không tự điền giá trị source bắt buộc. Thiếu source field vẫn được mang sang None để validator fail-closed.
    source_ref = record.get('source_record_reference')
    trace_material = {
        'context_version': CONTEXT_VERSION,
        'adapter_version': spec['adapter_version'],
        'source_artifact_sha256': spec['artifact_sha256'],
        'source_record_reference': source_ref,
        'model_version': record.get('model_version'),
    }
    trace_id = 'CTX-' + canonical_hash(trace_material)[:24].upper()
    flags = ['RETROSPECTIVE_OFFLINE']
    if source_key == 'credit':
        flags += ['ISSUE_MONTH_GRANULARITY', 'NOT_BANK_PD']
    else:
        flags += ['DGRAPHFIN_ANOMALY_LABEL', 'ASSESSMENT_TIME_NOT_AVAILABLE']

    return {
        'source_record_reference': None if source_ref is None else str(source_ref),
        'entity_scope': record.get('entity_scope'),
        'risk_domain': record.get('risk_domain'),
        'score_value': record.get('score_value'),
        'score_semantics': record.get('score_semantics'),
        'model_version': record.get('model_version'),
        'source_dataset': record.get('source_dataset'),
        'assessment_time': record.get('assessment_time'),
        'assessment_time_status': record.get('assessment_time_status'),
        'context_version': CONTEXT_VERSION,
        'adapter_version': spec['adapter_version'],
        'source_artifact_sha256': spec['artifact_sha256'],
        'trace_id': trace_id,
        'quality_flags': flags,
    }

def validate_context(context: dict, source_key: str, reference_available: bool = True) -> dict:
    spec = SOURCE_REGISTRY[source_key]
    missing = sorted(
        field for field in CONTEXT_REQUIRED
        if field not in context or (context[field] is None and field != 'assessment_time') or context[field] == ''
    )
    if missing:
        return {'status': 'REJECTED', 'route': 'NONE', 'reasons': [f'MISSING:{field}' for field in missing]}
    if not reference_available:
        return {'status': 'HELD', 'route': 'DATA_REVIEW', 'reasons': ['REFERENCE_UNAVAILABLE']}
    if 'joint_score' in context or 'joint_credit_fraud_score' in context:
        return {'status': 'REJECTED', 'route': 'NONE', 'reasons': ['UNAUTHORISED_JOINT_SCORE']}

    reasons = []
    try:
        score = float(context['score_value'])
    except (TypeError, ValueError):
        return {'status': 'REJECTED', 'route': 'NONE', 'reasons': ['SCORE_NOT_NUMERIC']}
    if not np.isfinite(score) or not 0.0 <= score <= 1.0:
        reasons.append('SCORE_OUT_OF_RANGE')
    if context['context_version'] != CONTEXT_VERSION:
        reasons.append('UNSUPPORTED_CONTEXT_VERSION')
    if context['entity_scope'] != spec['entity_scope']:
        reasons.append('ENTITY_SCOPE_MISMATCH')
    if context['risk_domain'] != spec['risk_domain']:
        reasons.append('DOMAIN_MISMATCH')
    if context['score_semantics'] != spec['score_semantics']:
        reasons.append('SCORE_SEMANTICS_MISMATCH')
    if context['model_version'] not in spec['model_versions']:
        reasons.append('MODEL_VERSION_UNREGISTERED')
    if context['source_dataset'] != spec['source_dataset']:
        reasons.append('SOURCE_DATASET_MISMATCH')
    if context['source_artifact_sha256'] != spec['artifact_sha256']:
        reasons.append('PROVENANCE_HASH_MISMATCH')
    if context['adapter_version'] != spec['adapter_version']:
        reasons.append('ADAPTER_VERSION_MISMATCH')
    if context['assessment_time_status'] != spec['assessment_time_status']:
        reasons.append('ASSESSMENT_TIME_STATUS_MISMATCH')
    if source_key == 'credit' and (context['assessment_time'] is None or context['assessment_time'] == ''):
        reasons.append('ASSESSMENT_TIME_REQUIRED')
    if source_key != 'credit' and context['assessment_time_status'] == 'not_available_in_source' and context['assessment_time'] not in [None, '']:
        reasons.append('ASSESSMENT_TIME_UNDECLARED_VALUE')

    if reasons:
        return {'status': 'REJECTED', 'route': 'NONE', 'reasons': reasons}
    return {'status': 'ACCEPTED', 'route': spec['route'], 'reasons': []}

def canonical_output(context: dict, validation: dict) -> dict:
    return {
        'trace_id': context['trace_id'],
        'status': validation['status'],
        'route': validation['route'],
        'reasons': sorted(validation['reasons']),
        'context_sha256': canonical_hash(context),
        'context_version': context['context_version'],
        'adapter_version': context['adapter_version'],
        'source_artifact_sha256': context['source_artifact_sha256'],
        'contract_sha256': CONTRACT_SHA256,
        'routing_policy_version': ROUTING_POLICY_VERSION,
        'routing_policy_sha256': ROUTING_POLICY_SHA256,
    }


In [ ]:
# Bulk conformance trên toàn bộ locked source outputs, báo riêng từng population.
bulk_summaries = {}
context_output_paths = {}

for source_key, spec in SOURCE_REGISTRY.items():
    frame = pd.read_parquet(spec['path'])
    contexts, result_rows = [], []
    preservation = {
        'score_value': 0, 'risk_domain': 0, 'score_semantics': 0,
        'entity_scope': 0, 'model_version': 0, 'source_dataset': 0,
        'source_record_reference': 0, 'assessment_time': 0, 'assessment_time_status': 0,
    }
    defaulted_mandatory_field_count = 0

    for record in frame.to_dict('records'):
        context = adapt_record(record, source_key)
        validation = validate_context(context, source_key)
        contexts.append(context)
        result_rows.append(canonical_output(context, validation))
        for field in preservation:
            left = context.get(field)
            right = record.get(field)
            if field == 'score_value':
                equal = (left is not None and right is not None and float(left) == float(right))
            else:
                equal = left == right
            preservation[field] += int(equal)

    n = len(frame)
    results = pd.DataFrame(result_rows)
    context_df = pd.DataFrame(contexts)
    out_path = RQ2_DIR / f'{source_key}_shared_contexts.parquet'
    context_df.to_parquet(out_path, index=False)
    context_output_paths[source_key] = out_path

    status_counts = results['status'].value_counts().to_dict() if n else {}
    bulk_summaries[source_key] = {
        'source_rows': n,
        'contexts_created': int(len(context_df)),
        'one_to_one_mapping_rate': None if n == 0 else float(len(context_df) / n),
        'accepted_count': int(status_counts.get('ACCEPTED', 0)),
        'held_count': int(status_counts.get('HELD', 0)),
        'rejected_count': int(status_counts.get('REJECTED', 0)),
        'schema_and_semantic_accept_rate': None if n == 0 else float(status_counts.get('ACCEPTED', 0) / n),
        'preservation_rates': {field: None if n == 0 else float(count / n) for field, count in preservation.items()},
        'defaulted_mandatory_field_count': defaulted_mandatory_field_count,
        'context_file': out_path.name,
        'context_file_sha256': sha256_file(out_path),
    }

(RQ2_DIR / 'rq2_bulk_conformance.json').write_text(json.dumps(bulk_summaries, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(bulk_summaries, ensure_ascii=False, indent=2))


In [ ]:
# Controlled invariant scenarios — không giả định cùng khách hàng thực.
def first_record(source_key: str) -> dict:
    return pd.read_parquet(SOURCE_REGISTRY[source_key]['path']).iloc[0].to_dict()

valid_credit = adapt_record(first_record('credit'), 'credit')
valid_node = adapt_record(first_record('fraud_node_only'), 'fraud_node_only')
valid_graph = adapt_record(first_record('fraud_node_plus_graph'), 'fraud_node_plus_graph')

cases = []
def add_case(case_id, context, source_key, expected_status, expected_route, reference_available=True):
    cases.append({
        'case_id': case_id, 'context': context, 'source_key': source_key,
        'reference_available': reference_available,
        'expected_status': expected_status, 'expected_route': expected_route,
    })

add_case('C01_VALID_CREDIT', deepcopy(valid_credit), 'credit', 'ACCEPTED', 'CREDIT_REVIEW')
add_case('C02_VALID_FRAUD_NODE', deepcopy(valid_node), 'fraud_node_only', 'ACCEPTED', 'FRAUD_GRAPH_REVIEW')
add_case('C03_VALID_FRAUD_GRAPH', deepcopy(valid_graph), 'fraud_node_plus_graph', 'ACCEPTED', 'FRAUD_GRAPH_REVIEW')
c = deepcopy(valid_credit); c.pop('score_semantics'); add_case('C04_MISSING_MANDATORY', c, 'credit', 'REJECTED', 'NONE')
c = deepcopy(valid_graph); c['score_semantics'] = 'generic_fraud_probability'; add_case('C05_WRONG_SEMANTICS', c, 'fraud_node_plus_graph', 'REJECTED', 'NONE')
c = deepcopy(valid_credit); c['model_version'] = 'unknown_model'; add_case('C06_UNSUPPORTED_MODEL', c, 'credit', 'REJECTED', 'NONE')
c = deepcopy(valid_credit); c['score_value'] = 1.01; add_case('C07_SCORE_RANGE', c, 'credit', 'REJECTED', 'NONE')
c = deepcopy(valid_node); c['source_artifact_sha256'] = '0'*64; add_case('C08_BAD_PROVENANCE', c, 'fraud_node_only', 'REJECTED', 'NONE')
add_case('C09_REFERENCE_UNAVAILABLE', deepcopy(valid_credit), 'credit', 'HELD', 'DATA_REVIEW', reference_available=False)
c = deepcopy(valid_credit); c['joint_credit_fraud_score'] = 0.5; add_case('C10_JOINT_SCORE_FORBIDDEN', c, 'credit', 'REJECTED', 'NONE')
c = deepcopy(valid_credit); c['risk_domain'] = 'fraud_graph'; add_case('C11_DOMAIN_MISMATCH', c, 'credit', 'REJECTED', 'NONE')
c = deepcopy(valid_graph); c['entity_scope'] = 'customer'; add_case('C12_ENTITY_SCOPE_ELEVATION', c, 'fraud_node_plus_graph', 'REJECTED', 'NONE')
c = deepcopy(valid_graph); c['context_version'] = 'unsupported_v999'; add_case('C13_UNSUPPORTED_CONTEXT_VERSION', c, 'fraud_node_plus_graph', 'REJECTED', 'NONE')

case_rows = []
for case in cases:
    result = validate_context(case['context'], case['source_key'], case['reference_available'])
    passed = result['status'] == case['expected_status'] and result['route'] == case['expected_route']
    case_rows.append({
        'case_id': case['case_id'], 'expected_status': case['expected_status'], 'actual_status': result['status'],
        'expected_route': case['expected_route'], 'actual_route': result['route'], 'pass': bool(passed),
        'reasons': json.dumps(result['reasons'], ensure_ascii=False),
    })
case_results = pd.DataFrame(case_rows)
case_results.to_csv(RQ2_DIR / 'rq2_controlled_cases.csv', index=False)
display(case_results)
if not bool(case_results['pass'].all()):
    raise AssertionError('Controlled RQ2 cases có ca FAIL.')

# Multi-context scenario is synthetic integration only: no same-customer or linkage claim.
def route_controlled_multi_context(context_specs):
    outputs = []
    for context, source_key in context_specs:
        val = validate_context(context, source_key)
        outputs.append(canonical_output(context, val))
    if any(o['status'] == 'REJECTED' for o in outputs):
        route = 'DATA_REVIEW'
    elif any(o['status'] == 'HELD' for o in outputs):
        route = 'DATA_REVIEW'
    else:
        domains = {c['risk_domain'] for c, _ in context_specs}
        route = 'MULTI_CONTEXT_REVIEW' if {'credit', 'fraud_graph'}.issubset(domains) else outputs[0]['route']
    result = {
        'case_reference_type': 'controlled_synthetic_scenario',
        'entity_linkage_status': 'synthetic_not_observed',
        'recommended_route': route,
        'input_trace_ids': sorted(o['trace_id'] for o in outputs),
        'joint_score': None,
        'routing_policy_version': ROUTING_POLICY_VERSION,
        'routing_policy_sha256': ROUTING_POLICY_SHA256,
    }
    result['output_sha256'] = canonical_hash(result)
    return result

multi_forward = route_controlled_multi_context([(deepcopy(valid_credit), 'credit'), (deepcopy(valid_graph), 'fraud_node_plus_graph')])
multi_reverse = route_controlled_multi_context([(deepcopy(valid_graph), 'fraud_node_plus_graph'), (deepcopy(valid_credit), 'credit')])
multi_context_pass = (
    multi_forward['recommended_route'] == 'MULTI_CONTEXT_REVIEW' and
    multi_forward['joint_score'] is None and
    multi_forward['entity_linkage_status'] == 'synthetic_not_observed' and
    multi_forward['output_sha256'] == multi_reverse['output_sha256']
)
(RQ2_DIR / 'rq2_multi_context_scenario.json').write_text(json.dumps({
    'status': 'PASS' if multi_context_pass else 'FAIL',
    'forward': multi_forward,
    'reverse': multi_reverse,
}, ensure_ascii=False, indent=2), encoding='utf-8')
if not multi_context_pass:
    raise AssertionError('Controlled multi-context scenario FAIL.')

# Determinism: same inputs + contract/policy -> same canonical output hash in 100 runs.
determinism_hashes = []
for _ in range(100):
    batch = []
    for context, source_key in [(deepcopy(valid_credit), 'credit'), (deepcopy(valid_node), 'fraud_node_only'), (deepcopy(valid_graph), 'fraud_node_plus_graph')]:
        result = validate_context(context, source_key)
        batch.append(canonical_output(context, result))
    batch = sorted(batch, key=lambda row: row['trace_id'])
    batch.append(route_controlled_multi_context([(deepcopy(valid_credit), 'credit'), (deepcopy(valid_graph), 'fraud_node_plus_graph')]))
    determinism_hashes.append(canonical_hash(batch))

determinism = {
    'repetitions': 100,
    'unique_output_hash_count': len(set(determinism_hashes)),
    'status': 'PASS' if len(set(determinism_hashes)) == 1 else 'FAIL',
    'order_invariance_for_independent_contexts': bool(multi_forward['output_sha256'] == multi_reverse['output_sha256']),
}
(RQ2_DIR / 'rq2_determinism.json').write_text(json.dumps(determinism, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'determinism': determinism, 'multi_context': multi_forward}, ensure_ascii=False, indent=2))


In [ ]:
# Final RQ2 evidence summary and gates.
def all_rates_one(summary: dict) -> bool:
    if summary['source_rows'] == 0:
        return False
    if summary['one_to_one_mapping_rate'] != 1.0:
        return False
    if summary['schema_and_semantic_accept_rate'] != 1.0:
        return False
    if summary['defaulted_mandatory_field_count'] != 0:
        return False
    return all(value == 1.0 for value in summary['preservation_rates'].values())

bulk_pass = all(all_rates_one(summary) for summary in bulk_summaries.values())
cases_pass = bool(case_results['pass'].all())
determinism_pass = determinism['status'] == 'PASS' and determinism['order_invariance_for_independent_contexts']
multi_context_pass = bool(multi_context_pass)
status = 'RQ2_EVIDENCE_PASS' if bulk_pass and cases_pass and determinism_pass and multi_context_pass else 'RQ2_EVIDENCE_FAIL'

summary = {
    'status': status,
    'contract_version': CONTEXT_VERSION,
    'contract_sha256': CONTRACT_SHA256,
    'routing_policy_version': ROUTING_POLICY_VERSION,
    'routing_policy_sha256': ROUTING_POLICY_SHA256,
    'bulk_conformance_pass': bulk_pass,
    'controlled_cases_pass': cases_pass,
    'controlled_case_count': int(len(case_results)),
    'multi_context_scenario_pass': multi_context_pass,
    'determinism': determinism,
    'claim_boundary': 'conformance to declared semantics and controlled routing only; no real Credit-Fraud customer linkage, no joint predictive uplift, no bank decision authority',
}
summary_path = RQ2_DIR / 'rq2_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
manifest = {
    'status': status,
    'summary_sha256': sha256_file(summary_path),
    'source_artifacts': {k: v['artifact_sha256'] for k, v in SOURCE_REGISTRY.items()},
    'context_outputs': {k: sha256_file(v) for k, v in context_output_paths.items()},
    'controlled_cases_sha256': sha256_file(RQ2_DIR / 'rq2_controlled_cases.csv'),
    'determinism_sha256': sha256_file(RQ2_DIR / 'rq2_determinism.json'),
    'bulk_conformance_sha256': sha256_file(RQ2_DIR / 'rq2_bulk_conformance.json'),
    'multi_context_scenario_sha256': sha256_file(RQ2_DIR / 'rq2_multi_context_scenario.json'),
    'contract_file_sha256': sha256_file(RQ2_DIR / 'shared_risk_context_contract_v2_1.json'),
    'routing_policy_file_sha256': sha256_file(RQ2_DIR / 'routing_policy_v2_1.json'),
}
(RQ2_DIR / 'rq2_evidence_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))
if status != 'RQ2_EVIDENCE_PASS':
    raise AssertionError('RQ2 không đạt acceptance criteria; không tự sửa output để đạt 100%.')


In [ ]:
# KLTN FINAL EVIDENCE LOCK v2.1
# Chạy SAU KHI:
#   - 01 đã có CREDIT_SOURCE_READY
#   - 02 đã có RQ1_FINAL_EVIDENCE_LOCKED
#   - 03 đã có RQ2_EVIDENCE_PASS
#
# Cell này KHÔNG huấn luyện lại, KHÔNG mở lại test, KHÔNG sửa evidence.
# Nó chỉ xác minh hash/status hiện có rồi tạo một lock manifest cuối cùng + SHA256 sidecar.

from pathlib import Path
import hashlib
import json
import os
from datetime import datetime, timezone

PROJECT_ROOT = Path(os.environ.get('KLTN_WORKSPACE', '/content/drive/MyDrive/KLTN_REPRODUCTION_25410008' if Path('/content/drive/MyDrive').is_dir() else str(Path.cwd() / 'kltn_workspace'))).resolve()
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'protocol_v2_1'

CREDIT_DIR = ARTIFACT_ROOT / 'credit'
RQ1_DIR = ARTIFACT_ROOT / 'rq1'
RQ2_DIR = ARTIFACT_ROOT / 'rq2'
FINAL_LOCK_DIR = ARTIFACT_ROOT / 'final_lock'
FINAL_LOCK_DIR.mkdir(parents=True, exist_ok=True)

LOCK_PATH = FINAL_LOCK_DIR / 'KLTN_FINAL_EVIDENCE_LOCK_v2.1.json'
LOCK_SHA_PATH = FINAL_LOCK_DIR / 'KLTN_FINAL_EVIDENCE_LOCK_v2.1.json.sha256'

# Không ghi đè một lock đã tồn tại.
if LOCK_PATH.exists() or LOCK_SHA_PATH.exists():
    raise RuntimeError(
        'REFUSED: KLTN_FINAL_EVIDENCE_LOCK_v2.1 đã tồn tại. '
        'Không ghi đè evidence lock. Nếu thật sự cần phiên bản mới, tăng version.'
    )

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def canonical_json_bytes(obj) -> bytes:
    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')

def canonical_hash(obj) -> str:
    return hashlib.sha256(canonical_json_bytes(obj)).hexdigest()

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f'MISSING REQUIRED FILE: {path}')
    return json.loads(path.read_text(encoding='utf-8'))

def require_file(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'MISSING REQUIRED FILE: {path}')
    if not path.is_file():
        raise RuntimeError(f'NOT A FILE: {path}')

def require_equal(actual, expected, message):
    if actual != expected:
        raise RuntimeError(
            f'INTEGRITY FAIL: {message}\n'
            f'  actual   = {actual}\n'
            f'  expected = {expected}'
        )

def verify_named_files(base_dir: Path, files_map: dict, label: str):
    checked = {}
    for name, expected_hash in files_map.items():
        path = base_dir / name
        require_file(path)
        actual_hash = sha256_file(path)
        require_equal(actual_hash, expected_hash, f'{label}: {name}')
        checked[name] = actual_hash
    return checked

# ---------------------------------------------------------------------
# 1. REQUIRED AUTHORITY FILES
# ---------------------------------------------------------------------
RUN_CONFIG_PATH = ARTIFACT_ROOT / 'run_config.json'
FEASIBILITY_PATH = ARTIFACT_ROOT / 'feasibility_report.json'

CREDIT_MANIFEST_PATH = CREDIT_DIR / 'credit_evidence_manifest.json'
CREDIT_FREEZE_PATH = CREDIT_DIR / 'credit_source_freeze_manifest.json'
CREDIT_METRICS_PATH = CREDIT_DIR / 'credit_metrics.json'

RQ1_FREEZE_PATH = RQ1_DIR / 'rq1_freeze_manifest.json'
RQ1_PRE_ACCESS_PATH = RQ1_DIR / 'rq1_pre_access_integrity.json'
RQ1_FINAL_ACCESS_PATH = RQ1_DIR / 'rq1_final_access_started.json'
RQ1_FINAL_MANIFEST_PATH = RQ1_DIR / 'rq1_final_manifest.json'
RQ1_SUMMARY_PATH = RQ1_DIR / 'rq1_summary.json'

RQ2_MANIFEST_PATH = RQ2_DIR / 'rq2_evidence_manifest.json'
RQ2_SUMMARY_PATH = RQ2_DIR / 'rq2_summary.json'
RQ2_CONTRACT_PATH = RQ2_DIR / 'shared_risk_context_contract_v2_1.json'
RQ2_POLICY_PATH = RQ2_DIR / 'routing_policy_v2_1.json'

required_paths = [
    RUN_CONFIG_PATH,
    FEASIBILITY_PATH,
    CREDIT_MANIFEST_PATH,
    CREDIT_FREEZE_PATH,
    CREDIT_METRICS_PATH,
    RQ1_FREEZE_PATH,
    RQ1_PRE_ACCESS_PATH,
    RQ1_FINAL_ACCESS_PATH,
    RQ1_FINAL_MANIFEST_PATH,
    RQ1_SUMMARY_PATH,
    RQ2_MANIFEST_PATH,
    RQ2_SUMMARY_PATH,
    RQ2_CONTRACT_PATH,
    RQ2_POLICY_PATH,
]

for path in required_paths:
    require_file(path)

# ---------------------------------------------------------------------
# 2. LOAD + STATUS GATES
# ---------------------------------------------------------------------
run_config = load_json(RUN_CONFIG_PATH)
feasibility = load_json(FEASIBILITY_PATH)

credit_manifest = load_json(CREDIT_MANIFEST_PATH)
credit_freeze = load_json(CREDIT_FREEZE_PATH)
credit_metrics = load_json(CREDIT_METRICS_PATH)

rq1_freeze = load_json(RQ1_FREEZE_PATH)
rq1_pre_access = load_json(RQ1_PRE_ACCESS_PATH)
rq1_final_access = load_json(RQ1_FINAL_ACCESS_PATH)
rq1_final_manifest = load_json(RQ1_FINAL_MANIFEST_PATH)
rq1_summary = load_json(RQ1_SUMMARY_PATH)

rq2_manifest = load_json(RQ2_MANIFEST_PATH)
rq2_summary = load_json(RQ2_SUMMARY_PATH)
rq2_contract_file = load_json(RQ2_CONTRACT_PATH)
rq2_policy_file = load_json(RQ2_POLICY_PATH)

require_equal(
    credit_manifest.get('status'),
    'CREDIT_SOURCE_READY',
    'Credit evidence status'
)
require_equal(
    rq1_freeze.get('status'),
    'RQ1_FREEZE_READY',
    'RQ1 freeze status'
)
require_equal(
    rq1_pre_access.get('status'),
    'RQ1_PRE_ACCESS_INTEGRITY_PASS',
    'RQ1 pre-access status'
)
require_equal(
    rq1_final_access.get('status'),
    'FINAL_ACCESS_STARTED',
    'RQ1 final-access status'
)
require_equal(
    rq1_final_manifest.get('status'),
    'RQ1_FINAL_EVIDENCE_LOCKED',
    'RQ1 final evidence status'
)
require_equal(
    rq1_summary.get('status'),
    'RQ1_FINAL_COMPLETE',
    'RQ1 summary status'
)
require_equal(
    rq2_manifest.get('status'),
    'RQ2_EVIDENCE_PASS',
    'RQ2 evidence manifest status'
)
require_equal(
    rq2_summary.get('status'),
    'RQ2_EVIDENCE_PASS',
    'RQ2 summary status'
)

# ---------------------------------------------------------------------
# 3. VERIFY CREDIT EVIDENCE CHAIN
# ---------------------------------------------------------------------
require_equal(
    sha256_file(CREDIT_FREEZE_PATH),
    credit_manifest.get('freeze_manifest_sha256'),
    'Credit freeze manifest hash'
)

credit_checked_files = verify_named_files(
    CREDIT_DIR,
    credit_manifest.get('files', {}),
    'Credit evidence'
)

# ---------------------------------------------------------------------
# 4. VERIFY RQ1 EVIDENCE CHAIN
# ---------------------------------------------------------------------
require_equal(
    sha256_file(RQ1_FREEZE_PATH),
    rq1_final_manifest.get('freeze_manifest_sha256'),
    'RQ1 freeze manifest hash'
)

require_equal(
    sha256_file(RQ1_PRE_ACCESS_PATH),
    rq1_final_manifest.get('pre_access_integrity_sha256'),
    'RQ1 pre-access record hash'
)

require_equal(
    sha256_file(RQ1_FINAL_ACCESS_PATH),
    rq1_final_manifest.get('final_access_record_sha256'),
    'RQ1 final-access record hash'
)

rq1_checked_files = verify_named_files(
    RQ1_DIR,
    rq1_final_manifest.get('files', {}),
    'RQ1 final evidence'
)

# Cross-check summary embedded in final manifest.
if rq1_final_manifest.get('summary') != rq1_summary:
    raise RuntimeError(
        'INTEGRITY FAIL: rq1_final_manifest.summary khác rq1_summary.json'
    )

# ---------------------------------------------------------------------
# 5. VERIFY RQ2 EVIDENCE CHAIN
# ---------------------------------------------------------------------
require_equal(
    sha256_file(RQ2_SUMMARY_PATH),
    rq2_manifest.get('summary_sha256'),
    'RQ2 summary hash'
)

rq2_named_checks = {
    'rq2_controlled_cases.csv': rq2_manifest.get('controlled_cases_sha256'),
    'rq2_determinism.json': rq2_manifest.get('determinism_sha256'),
    'rq2_bulk_conformance.json': rq2_manifest.get('bulk_conformance_sha256'),
    'rq2_multi_context_scenario.json': rq2_manifest.get('multi_context_scenario_sha256'),
    'shared_risk_context_contract_v2_1.json': rq2_manifest.get('contract_file_sha256'),
    'routing_policy_v2_1.json': rq2_manifest.get('routing_policy_file_sha256'),
}

for name, expected_hash in rq2_named_checks.items():
    if not expected_hash:
        raise RuntimeError(f'RQ2 manifest thiếu expected hash cho {name}')
    path = RQ2_DIR / name
    require_file(path)
    require_equal(
        sha256_file(path),
        expected_hash,
        f'RQ2 evidence: {name}'
    )

# Verify RQ2 context outputs.
rq2_context_outputs = rq2_manifest.get('context_outputs', {})
for source_key, expected_hash in rq2_context_outputs.items():
    path = RQ2_DIR / f'{source_key}_shared_contexts.parquet'
    require_file(path)
    require_equal(
        sha256_file(path),
        expected_hash,
        f'RQ2 context output: {source_key}'
    )

# Verify RQ2 source artifacts against exact upstream prediction files.
rq2_source_paths = {
    'credit': CREDIT_DIR / 'credit_predictions_for_context.parquet',
    'fraud_node_only': RQ1_DIR / 'dgraphfin_node_only_predictions_for_context.parquet',
    'fraud_node_plus_graph': RQ1_DIR / 'dgraphfin_node_plus_graph_predictions_for_context.parquet',
}

for source_key, expected_hash in rq2_manifest.get('source_artifacts', {}).items():
    if source_key not in rq2_source_paths:
        raise RuntimeError(f'Unknown RQ2 source_artifact key: {source_key}')
    path = rq2_source_paths[source_key]
    require_file(path)
    require_equal(
        sha256_file(path),
        expected_hash,
        f'RQ2 source artifact: {source_key}'
    )

# Contract/policy semantic hashes reported in RQ2 summary.
require_equal(
    rq2_summary.get('contract_sha256'),
    rq2_contract_file.get('contract_sha256'),
    'RQ2 contract semantic SHA'
)
require_equal(
    rq2_summary.get('routing_policy_sha256'),
    rq2_policy_file.get('policy_sha256'),
    'RQ2 routing-policy semantic SHA'
)

# Explicit acceptance criteria from RQ2 summary.
require_equal(
    rq2_summary.get('bulk_conformance_pass'),
    True,
    'RQ2 bulk conformance'
)
require_equal(
    rq2_summary.get('controlled_cases_pass'),
    True,
    'RQ2 controlled cases'
)
require_equal(
    rq2_summary.get('multi_context_scenario_pass'),
    True,
    'RQ2 multi-context scenario'
)
require_equal(
    rq2_summary.get('determinism', {}).get('status'),
    'PASS',
    'RQ2 determinism'
)
require_equal(
    rq2_summary.get('determinism', {}).get('unique_output_hash_count'),
    1,
    'RQ2 unique deterministic output hash count'
)
require_equal(
    rq2_summary.get('determinism', {}).get(
        'order_invariance_for_independent_contexts'
    ),
    True,
    'RQ2 order invariance'
)

# ---------------------------------------------------------------------
# 6. BUILD IMMUTABLE EVIDENCE INVENTORY
# ---------------------------------------------------------------------
# Chỉ inventory evidence thực nghiệm:
# - run_config + feasibility
# - toàn bộ credit/
# - toàn bộ rq1/
# - toàn bộ rq2/
# Không đưa cache/ vào root lock vì cache là artifact dẫn xuất/trung gian.
# Không đưa final_lock/ vào chính inventory để tránh circular dependency.

inventory_paths = [RUN_CONFIG_PATH, FEASIBILITY_PATH]

for directory in [CREDIT_DIR, RQ1_DIR, RQ2_DIR]:
    inventory_paths.extend(
        sorted(
            p for p in directory.rglob('*')
            if p.is_file()
        )
    )

# Deduplicate by resolved string.
unique_paths = {}
for path in inventory_paths:
    unique_paths[str(path.resolve())] = path

inventory = []
for path in sorted(unique_paths.values(), key=lambda p: str(p.relative_to(ARTIFACT_ROOT))):
    rel = str(path.relative_to(ARTIFACT_ROOT))
    inventory.append({
        'relative_path': rel,
        'size_bytes': int(path.stat().st_size),
        'sha256': sha256_file(path),
    })

evidence_root_digest = canonical_hash(inventory)

# ---------------------------------------------------------------------
# 7. FINAL LOCK OBJECT
# ---------------------------------------------------------------------
lock = {
    'lock_version': 'KLTN_FINAL_EVIDENCE_LOCK_v2.1',
    'status': 'KLTN_FINAL_EVIDENCE_LOCKED',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'protocol_version': run_config.get('protocol_version'),
    'artifact_root': str(ARTIFACT_ROOT),

    'authority_status': {
        'credit': credit_manifest.get('status'),
        'rq1': rq1_final_manifest.get('status'),
        'rq2': rq2_manifest.get('status'),
    },

    'authority_manifests': {
        'run_config': {
            'relative_path': str(RUN_CONFIG_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(RUN_CONFIG_PATH),
        },
        'feasibility_report': {
            'relative_path': str(FEASIBILITY_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(FEASIBILITY_PATH),
        },
        'credit_evidence_manifest': {
            'relative_path': str(CREDIT_MANIFEST_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(CREDIT_MANIFEST_PATH),
        },
        'credit_source_freeze_manifest': {
            'relative_path': str(CREDIT_FREEZE_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(CREDIT_FREEZE_PATH),
        },
        'rq1_freeze_manifest': {
            'relative_path': str(RQ1_FREEZE_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(RQ1_FREEZE_PATH),
        },
        'rq1_pre_access_integrity': {
            'relative_path': str(RQ1_PRE_ACCESS_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(RQ1_PRE_ACCESS_PATH),
        },
        'rq1_final_access_started': {
            'relative_path': str(RQ1_FINAL_ACCESS_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(RQ1_FINAL_ACCESS_PATH),
        },
        'rq1_final_manifest': {
            'relative_path': str(RQ1_FINAL_MANIFEST_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(RQ1_FINAL_MANIFEST_PATH),
        },
        'rq2_evidence_manifest': {
            'relative_path': str(RQ2_MANIFEST_PATH.relative_to(ARTIFACT_ROOT)),
            'sha256': sha256_file(RQ2_MANIFEST_PATH),
        },
    },

    'rq1_result': {
        'status': rq1_summary.get('status'),
        'primary_seed': rq1_summary.get('primary_seed'),
        'test_rows': rq1_summary.get('test_rows'),
        'delta_ap': rq1_summary.get('delta_ap'),
        'delta_ap_ci_95': rq1_summary.get('delta_ap_ci_95'),
        'bootstrap_requested_evaluable': rq1_summary.get(
            'bootstrap_requested_evaluable'
        ),
        'bootstrap_evaluable': rq1_summary.get('bootstrap_evaluable'),
        'bootstrap_seed': rq1_summary.get('bootstrap_seed'),
        'classification': rq1_summary.get('classification'),
        'interpretation_scope': rq1_summary.get('interpretation_scope'),
    },

    'rq2_result': {
        'status': rq2_summary.get('status'),
        'contract_version': rq2_summary.get('contract_version'),
        'contract_sha256': rq2_summary.get('contract_sha256'),
        'routing_policy_version': rq2_summary.get('routing_policy_version'),
        'routing_policy_sha256': rq2_summary.get('routing_policy_sha256'),
        'bulk_conformance_pass': rq2_summary.get('bulk_conformance_pass'),
        'controlled_cases_pass': rq2_summary.get('controlled_cases_pass'),
        'controlled_case_count': rq2_summary.get('controlled_case_count'),
        'multi_context_scenario_pass': rq2_summary.get(
            'multi_context_scenario_pass'
        ),
        'determinism': rq2_summary.get('determinism'),
        'claim_boundary': rq2_summary.get('claim_boundary'),
    },

    'credit_result': {
        'status': credit_manifest.get('status'),
        'metrics': credit_metrics,
        'claim_boundary': (
            'independent credit-source model used to produce Credit Context; '
            'not a joint Credit-Fraud model and not a claim of bank-policy validity'
        ),
    },

    'evidence_inventory_count': len(inventory),
    'evidence_inventory': inventory,
    'evidence_root_digest_sha256': evidence_root_digest,

    'global_claim_boundary': (
        'This lock attests integrity and internal consistency of the declared '
        'KLTN evidence chain only. It does not establish real Credit-Fraud '
        'customer linkage, joint predictive uplift, production effectiveness, '
        'bank decision authority, or compliance with a complete governance framework.'
    ),

    'mutation_policy': (
        'DO_NOT_OVERWRITE. Any post-lock technical correction or additional '
        'analysis must use a new version and document whether predictions, '
        'metrics, semantic contract, routing policy, or claim scope changed.'
    ),
}

# ---------------------------------------------------------------------
# 8. WRITE LOCK ONCE + SIDECAR SHA256
# ---------------------------------------------------------------------
LOCK_PATH.write_text(
    json.dumps(lock, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

lock_sha256 = sha256_file(LOCK_PATH)

LOCK_SHA_PATH.write_text(
    f'{lock_sha256}  {LOCK_PATH.name}\n',
    encoding='utf-8',
)

# Verify immediately after writing.
require_equal(
    sha256_file(LOCK_PATH),
    lock_sha256,
    'Final lock self-verification'
)

print('============================================================')
print('KLTN FINAL EVIDENCE LOCK v2.1 — PASS')
print('============================================================')
print('STATUS:', lock['status'])
print('EVIDENCE FILE COUNT:', len(inventory))
print('EVIDENCE ROOT DIGEST:', evidence_root_digest)
print('FINAL LOCK SHA256:', lock_sha256)
print('LOCK FILE:', LOCK_PATH)
print('SHA SIDECAR:', LOCK_SHA_PATH)
print()
print('RQ1:', rq1_summary.get('classification'))
print('  Delta AP:', rq1_summary.get('delta_ap'))
print('  95% CI:', rq1_summary.get('delta_ap_ci_95'))
print('RQ2:', rq2_summary.get('status'))
print('  Controlled cases:', rq2_summary.get('controlled_case_count'))
print('  Determinism:', rq2_summary.get('determinism'))
print()
print('DO NOT RERUN OR OVERWRITE LOCKED EVIDENCE WITHOUT A NEW VERSION.')
